In [ ]:
import scanpy as sc
import anndata
import numpy as np
import pandas as pd
import os
from scipy import sparse
from scipy import io
import seaborn as sns

In [ ]:
path="PATH"
os.chdir(path)

In [ ]:

counts=io.mmread('./counts.mtx')
counts=sparse.csr_matrix(counts)
barcodes=pd.read_csv('./barcodes.csv')
genes=pd.read_csv('./genes.csv')
adata=sc.AnnData(counts.T)
adata.obs_names=barcodes['Barcode'].values
adata.var_names=genes['Gene'].values


In [ ]:

cellMeta=pd.read_csv("./cellMeta.csv",index_col=0)
adata.obs=cellMeta


In [11]:
results_file = 'adata_raw.h5ad' 
adata.write(results_file)

In [ ]:
adata = sc.read_h5ad('adata_raw.h5ad')

In [13]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')

In [14]:
adata.var_names_make_unique()  # this is unnecessary if using `var_names='gene_ids'` in `sc.read_10x_mtx`
adata

AnnData object with n_obs × n_vars = 100726 × 60642
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'RNA_snn_res.1', 'seurat_clusters', 'cell_type', 'sample', 'cytopenia', 'broad_ct'

In [15]:
sc.pp.normalize_total(adata, target_sum=1e4)

normalizing counts per cell
    finished (0:00:00)


In [16]:
sc.pp.log1p(adata)

In [ ]:
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)


In [ ]:
sc.pp.scale(adata, max_value=10)


In [ ]:
sc.tl.pca(adata, svd_solver='arpack')


In [ ]:
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=70)


In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=50)


In [ ]:
#replace umap coordinates with that of VarID
X_umap = pd.read_csv('./umap',header=None)
X_umap = X_umap[[1,2]]
X_umap = X_umap.tail(-1)
X_umap = X_umap.astype('float')
adata.obsm['X_umap'] = X_umap.values
adata.obsm['X_umap']
sc.pl.embedding(adata, basis='X_umap')


In [ ]:
cpart = pd.read_csv('./cpart',header=None)

In [ ]:
cpart.astype('category')

In [ ]:
cpart = cpart.tail(-1)

In [ ]:
cpart = cpart.astype('category')

In [ ]:
cpart = pd.Series(cpart[1].values, index=cpart[0])

In [ ]:
adata.obs['cpart'] = cpart

In [ ]:
sc.pl.umap(adata, color=['cpart'])

In [ ]:
sc.tl.leiden(adata)

In [ ]:
sc.pl.umap(adata, color=['cell_type'])

In [ ]:
results_file = './GSE288357_scanpy_obj.h5ad' 
adata.write(results_file)

In [ ]:
#computation of mes infl score
sig_genes_pro_select = ["CCL2", "CXCL8", "CXCL3", "IL6", "HMOX1","CXCL2", "CXCL5", "PTGS2"]

overlap_sig_genes_pro_select = list(set(sig_genes_pro_select) & set(adata.var_names))

sc.tl.score_genes(adata, overlap_sig_genes_pro_select, score_name='infl.mes.sig.pro.select score')
adata[adata.obs['cell_type'].isin(['Infl.mesenchymal','Mesenchymal','myeloma.mesenchymal'])].obs['infl.mes.sig.pro.select score'].to_csv('infl_mes_sig_pro_sel_score')
